# 02 — Context Managers

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- comprendre le protocole `with` et pourquoi il existe
- implémenter `__enter__` et `__exit__` dans une classe
- utiliser `@contextmanager` de `contextlib` pour un CM en générateur
- gérer les exceptions dans `__exit__`
- utiliser `ExitStack` pour empiler des context managers dynamiquement
- connaître les CMs courants de la stdlib (`suppress`, `redirect_stdout`, etc.)

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- toute l'Initiation 5j : types, fonctions typées, fichiers, exceptions
- le modèle objet (classes, `__init__`, `__repr__`)
- les décorateurs et `functools.wraps` (notebook 01 de cette section)
- les générateurs (`yield`)

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- les design patterns (section 05)
- le logging (section 08) — on utilise `print` pour illustrer
- l'async (`async with`) — hors programme intermédiaire

## Plan

1. Le problème : nettoyage garanti
2. Le protocole `with`
3. Implémenter `__enter__` / `__exit__`
4. Gestion d'exceptions dans `__exit__`
5. `@contextmanager` : le raccourci générateur
6. Context managers imbriqués
7. `ExitStack` : empiler dynamiquement
8. CMs utiles de la stdlib
9. Pièges et bonnes pratiques
10. Synthèse
11. Exercices

---

## 1. Le problème : nettoyage garanti

Quand on ouvre une ressource (fichier, connexion, verrou), il faut la fermer **quoi qu'il arrive** — même en cas d'exception. Sans `with`, on écrit :

In [ ]:
f = open("/tmp/demo_cm.txt", "w")
try:
    f.write("Hello")
finally:
    f.close()


C'est verbeux et facile à oublier. Le `with` remplace ce pattern :

In [ ]:
with open("/tmp/demo_cm.txt", "w") as f:
    f.write("Hello")
# f.close() est appelé automatiquement, même si une exception survient


Le `with` appelle deux méthodes magiques sur l'objet :

1. `__enter__()` au début du bloc → retourne la valeur assignée à `as`
2. `__exit__()` à la fin du bloc → nettoie, même en cas d'exception

---

## 2. Le protocole `with`

Tout objet qui implémente `__enter__` et `__exit__` peut être utilisé avec `with`. On appelle cet objet un **context manager** (CM).

In [ ]:
class MonCM:
    def __enter__(self):
        print("Entrée dans le bloc with")
        return self  # valeur de 'as'

    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Sortie du bloc with")
        return False  # ne pas supprimer l'exception


In [ ]:
with MonCM() as cm:
    print("Dans le bloc")
    print(f"cm = {cm}")


### Les paramètres de `__exit__`

| Paramètre | Valeur si pas d'exception | Valeur si exception |
|-----------|---------------------------|---------------------|
| `exc_type` | `None` | La classe de l'exception |
| `exc_val` | `None` | L'instance de l'exception |
| `exc_tb` | `None` | Le traceback |

Si `__exit__` retourne `True`, l'exception est **supprimée** (avalée). Si elle retourne `False` ou `None`, l'exception se propage normalement.

---

## 3. Implémenter `__enter__` / `__exit__`

Construisons un CM concret : un chronomètre qui mesure le temps dans le bloc.

In [ ]:
import time

class Chronometre:
    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.perf_counter() - self.start
        print(f"Temps écoulé : {self.elapsed:.4f}s")
        return False


In [ ]:
with Chronometre() as c:
    total = sum(range(1_000_000))

print(f"Résultat : {total}, en {c.elapsed:.4f}s")


### CM pour une connexion fictive

In [ ]:
class Connexion:
    def __init__(self, url: str) -> None:
        self.url = url
        self.active = False

    def __enter__(self):
        print(f"Connexion à {self.url}")
        self.active = True
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"Déconnexion de {self.url}")
        self.active = False
        return False

    def requete(self, sql: str) -> str:
        if not self.active:
            raise RuntimeError("Connexion fermée")
        return f"Résultat de '{sql}'"


In [ ]:
with Connexion("localhost:5432") as conn:
    print(conn.requete("SELECT 1"))

print(f"Encore active ? {conn.active}")


---

## 4. Gestion d'exceptions dans `__exit__`

Le nettoyage se fait **même en cas d'exception**. C'est tout l'intérêt du CM.

In [ ]:
class CMVerbose:
    def __enter__(self):
        print("ENTER")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type:
            print(f"EXIT avec exception : {exc_type.__name__}: {exc_val}")
        else:
            print("EXIT normal")
        return False  # on ne supprime PAS l'exception


In [ ]:
try:
    with CMVerbose():
        print("Travail...")
        raise ValueError("Erreur simulée")
except ValueError as e:
    print(f"Exception récupérée : {e}")


### Supprimer une exception

In [ ]:
class CMSuppresseur:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is ZeroDivisionError:
            print("Division par zéro interceptée et supprimée")
            return True  # exception avalée
        return False


In [ ]:
with CMSuppresseur():
    print(1 / 0)  # pas d'exception propagée

print("Exécution continue normalement")


**Attention :** supprimer des exceptions est dangereux. Ne le faites que pour des cas très spécifiques (comme `contextlib.suppress`).

---

## 5. `@contextmanager` : le raccourci générateur

Écrire une classe avec `__enter__`/`__exit__` est verbeux. `contextlib.contextmanager` transforme un simple **générateur** en CM.

In [ ]:
from contextlib import contextmanager

@contextmanager
def chrono():
    start = time.perf_counter()
    yield  # le code du bloc with s'exécute ici
    elapsed = time.perf_counter() - start
    print(f"Temps : {elapsed:.4f}s")


In [ ]:
with chrono():
    total = sum(range(500_000))


### Avec une valeur retournée (`yield val`)

In [ ]:
@contextmanager
def fichier_temp(contenu: str):
    import tempfile, os
    fd, path = tempfile.mkstemp(suffix=".txt")
    try:
        with os.fdopen(fd, "w") as f:
            f.write(contenu)
        yield path  # valeur de 'as'
    finally:
        os.unlink(path)
        print(f"Fichier {path} supprimé")


In [ ]:
with fichier_temp("Bonjour !") as chemin:
    print(f"Fichier créé : {chemin}")
    with open(chemin) as f:
        print(f.read())

# Le fichier a été supprimé automatiquement


### Structure du générateur CM

```python
@contextmanager
def mon_cm():
    # __enter__ : initialisation
    resource = acquire()
    try:
        yield resource  # code du bloc with
    finally:
        # __exit__ : nettoyage
        release(resource)
```

Le `try/finally` dans le générateur garantit le nettoyage même en cas d'exception.

### Gestion d'exceptions dans le générateur

In [ ]:
@contextmanager
def catch_and_log():
    try:
        yield
    except Exception as e:
        print(f"Exception capturée : {type(e).__name__}: {e}")
        # Ne pas re-raise = exception supprimée


In [ ]:
with catch_and_log():
    raise RuntimeError("Boom")

print("Tout va bien")


---

## 6. Context managers imbriqués

On peut imbriquer des `with` ou les combiner sur une seule ligne.

In [ ]:
with open("/tmp/a.txt", "w") as a, open("/tmp/b.txt", "w") as b:
    a.write("fichier A")
    b.write("fichier B")


Depuis Python 3.10, on peut utiliser les parenthèses pour plus de lisibilité :

In [ ]:
with (
    open("/tmp/a.txt", "w") as a,
    open("/tmp/b.txt", "w") as b,
):
    a.write("fichier A")
    b.write("fichier B")


---

## 7. `ExitStack` : empiler dynamiquement

Quand le nombre de CMs n'est pas connu à l'avance, `ExitStack` permet de les empiler dans une boucle.

In [ ]:
from contextlib import ExitStack

noms_fichiers = ["/tmp/f1.txt", "/tmp/f2.txt", "/tmp/f3.txt"]

with ExitStack() as stack:
    fichiers = [
        stack.enter_context(open(nom, "w"))
        for nom in noms_fichiers
    ]
    for i, f in enumerate(fichiers):
        f.write(f"Contenu {i}")

# Tous les fichiers sont fermés à la sortie du with
print("Fichiers fermés")


### `ExitStack` avec des callbacks

In [ ]:
from contextlib import ExitStack

with ExitStack() as stack:
    stack.callback(print, "Nettoyage 3")
    stack.callback(print, "Nettoyage 2")
    stack.callback(print, "Nettoyage 1")
    print("Travail en cours...")
# Les callbacks s'exécutent en ordre LIFO (pile)


### Transférer la responsabilité avec `pop_all`

In [ ]:
from contextlib import ExitStack

def ouvrir_fichiers(noms: list[str]) -> tuple[list, ExitStack]:
    """Ouvre des fichiers et transfère la responsabilité de fermeture."""
    stack = ExitStack()
    try:
        fichiers = [stack.enter_context(open(n, "w")) for n in noms]
        # Succès : on transfère la responsabilité au caller
        return fichiers, stack.pop_all()
    except:
        stack.close()  # Échec : on nettoie tout
        raise

fichiers, cleanup = ouvrir_fichiers(["/tmp/x1.txt", "/tmp/x2.txt"])
# ... utiliser fichiers ...
cleanup.close()  # ferme tout quand on veut
print("Fichiers fermés manuellement")


---

## 8. CMs utiles de la stdlib

La bibliothèque standard fournit plusieurs context managers prêts à l'emploi.

| Module | CM | Usage |
|--------|----|-------|
| `contextlib` | `suppress(*exc)` | Ignorer certaines exceptions |
| `contextlib` | `redirect_stdout(f)` | Rediriger stdout vers un fichier |
| `contextlib` | `redirect_stderr(f)` | Rediriger stderr |
| `contextlib` | `closing(thing)` | Appeler `.close()` à la fin |
| `contextlib` | `nullcontext(val)` | CM qui ne fait rien |
| `tempfile` | `TemporaryDirectory()` | Répertoire temporaire auto-supprimé |
| `tempfile` | `NamedTemporaryFile()` | Fichier temporaire auto-supprimé |
| `decimal` | `localcontext()` | Contexte décimal temporaire |
| `unittest.mock` | `patch(...)` | Mock temporaire |

### `suppress` : ignorer des exceptions

In [ ]:
from contextlib import suppress
import os

# Sans suppress :
# try:
#     os.remove("/tmp/inexistant.txt")
# except FileNotFoundError:
#     pass

# Avec suppress :
with suppress(FileNotFoundError):
    os.remove("/tmp/inexistant.txt")

print("Pas d'erreur")


### `redirect_stdout` : capturer la sortie

In [ ]:
from contextlib import redirect_stdout
from io import StringIO

buffer = StringIO()
with redirect_stdout(buffer):
    print("Ce texte va dans le buffer")

print(f"Capturé : {buffer.getvalue()!r}")


### `TemporaryDirectory` : dossier auto-nettoyé

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path

with TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "hello.txt"
    path.write_text("Bonjour")
    print(f"Fichier : {path}")
    print(f"Existe : {path.exists()}")

print(f"Après with, existe : {Path(tmpdir).exists()}")


### `nullcontext` : CM conditionnel

In [ ]:
from contextlib import nullcontext

def traiter(fichier: str | None = None):
    cm = open(fichier, "w") if fichier else nullcontext()
    with cm as f:
        if f:
            print("Écriture dans le fichier")
        else:
            print("Pas de fichier, traitement en mémoire")

traiter()  # nullcontext
traiter("/tmp/out.txt")  # vrai fichier


---

## 9. Pièges et bonnes pratiques

Erreurs fréquentes avec les context managers.

### Piège 1 : oublier le `yield` dans `@contextmanager`

In [ ]:
from contextlib import contextmanager

# @contextmanager
# def mauvais():
#     print("setup")
#     # pas de yield → RuntimeError à l'usage
#     print("cleanup")
print("Toujours mettre un yield, même sans valeur")


### Piège 2 : ne pas protéger le `yield` avec `try/finally`

In [ ]:
@contextmanager
def fragile():
    print("Setup")
    yield  # si une exception arrive, le cleanup ne se fait pas !
    print("Cleanup")  # jamais atteint en cas d'exception

try:
    with fragile():
        raise ValueError("Boom")
except ValueError:
    print("Cleanup n'a PAS été exécuté !")


**Solution :** toujours protéger avec `try/finally` :

In [ ]:
@contextmanager
def robuste():
    print("Setup")
    try:
        yield
    finally:
        print("Cleanup")  # toujours exécuté

try:
    with robuste():
        raise ValueError("Boom")
except ValueError:
    print("Cleanup a bien été exécuté")


### Piège 3 : confondre `__enter__` et `__init__`

L'initialisation de la ressource doit se faire dans `__enter__`, pas dans `__init__`. Sinon, si on crée l'objet sans `with`, la ressource est ouverte mais jamais fermée.

---

## Synthèse

| Concept | Syntaxe | À retenir |
|---------|---------|----------|
| Protocole CM | `__enter__` / `__exit__` | `with obj as val:` |
| Raccourci | `@contextmanager` + `yield` | Générateur → CM |
| `ExitStack` | `stack.enter_context(cm)` | Empiler des CMs dynamiquement |
| `suppress` | `with suppress(Exc):` | Ignorer une exception |
| `redirect_stdout` | `with redirect_stdout(f):` | Capturer stdout |
| Retour de `__exit__` | `True` → avale l'exception | `False`/`None` → propage |

### Règles à retenir

1. Toute ressource acquise doit être libérée → utiliser un CM.
2. Toujours protéger le `yield` avec `try/finally` dans `@contextmanager`.
3. Initialiser la ressource dans `__enter__`, pas dans `__init__`.
4. `__exit__` retourne `True` uniquement si l'exception doit être avalée.
5. `ExitStack` quand le nombre de CMs est dynamique.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — CM indentation logger *(facile)*

Écrivez un context manager `indent_log` qui affiche `>>> Début section` à l'entrée et `<<< Fin section` à la sortie. Utilisez `@contextmanager`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Context_managers", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from contextlib import contextmanager

@contextmanager
def indent_log(section: str):
    print(f">>> Début {section}")
    try:
        yield
    finally:
        print(f"<<< Fin {section}")

with indent_log("traitement"):
    print("  Calcul en cours...")
```

</details>

### Exercice 2 — CM changeur de répertoire *(facile)*

Écrivez un CM `change_dir(path)` qui change le répertoire courant (`os.chdir`) au début du bloc et restaure l'ancien à la fin.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Context_managers", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import os
from contextlib import contextmanager

@contextmanager
def change_dir(path: str):
    ancien = os.getcwd()
    os.chdir(path)
    try:
        yield path
    finally:
        os.chdir(ancien)

print(f"Avant : {os.getcwd()}")
with change_dir("/tmp") as p:
    print(f"Dans : {os.getcwd()}")
print(f"Après : {os.getcwd()}")
```

</details>

### Exercice 3 — CM transaction fictive *(moyen)*

Créez une classe `Transaction` qui implémente le protocole CM :

- `__enter__` : affiche 'BEGIN', retourne `self`
- si pas d'exception, `__exit__` affiche 'COMMIT'
- si exception, `__exit__` affiche 'ROLLBACK' et laisse l'exception se propager

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Context_managers", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
class Transaction:
    def __enter__(self):
        print("BEGIN")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type:
            print("ROLLBACK")
        else:
            print("COMMIT")
        return False

with Transaction():
    print("INSERT ...")

try:
    with Transaction():
        raise RuntimeError("Erreur")
except RuntimeError:
    print("Exception propagée")
```

</details>

### Exercice 4 — Pool de connexions avec ExitStack *(moyen)*

Simulez un pool de connexions.

- Classe `FakeConn(name)` avec `__enter__`/`__exit__` qui print 'open name' / 'close name'
- Utilisez `ExitStack` pour ouvrir N connexions et les fermer toutes

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Context_managers", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
from contextlib import ExitStack

class FakeConn:
    def __init__(self, name: str) -> None:
        self.name = name

    def __enter__(self):
        print(f"open {self.name}")
        return self

    def __exit__(self, *args):
        print(f"close {self.name}")
        return False

noms = ["db1", "db2", "db3"]
with ExitStack() as stack:
    conns = [stack.enter_context(FakeConn(n)) for n in noms]
    print(f"Toutes ouvertes : {[c.name for c in conns]}")
```

</details>

### Exercice 5 — CM réessai avec backoff *(difficile)*

Écrivez un CM `retry_block(max_retries, backoff)` qui :

- intercepte les exceptions dans le bloc
- réessaie le bloc jusqu'à `max_retries` fois
- attend `backoff * attempt` secondes entre chaque tentative
- lève l'exception après épuisement des tentatives

Indice : c'est plus simple avec une boucle et `@contextmanager`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Context_managers", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
import time
from contextlib import contextmanager

def retry_block(max_retries: int = 3, backoff: float = 0.1):
    """Utilisation : for attempt in retry_block(3):
                       with attempt:
                           ..."""
    # On utilise un pattern itérateur + CM
    for i in range(max_retries):
        try:
            yield i
            return  # succès
        except Exception as e:
            if i == max_retries - 1:
                raise
            time.sleep(backoff * (i + 1))
            print(f"Tentative {i+1} échouée : {e}")
```

</details>

### Exercice 6 — CM Atomic File Write *(difficile)*

Écrivez un CM `atomic_write(path)` qui :

- écrit dans un fichier temporaire dans le même répertoire
- si le bloc se termine sans exception, renomme le fichier temporaire vers `path`
- si une exception survient, supprime le fichier temporaire

Cela garantit que `path` contient soit l'ancien contenu, soit le nouveau complet.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Context_managers", exercice=6)


<details>
<summary>📖 Voir la correction</summary>

```python
import os
import tempfile
from contextlib import contextmanager
from pathlib import Path

@contextmanager
def atomic_write(path: str):
    target = Path(path)
    fd, tmp_path = tempfile.mkstemp(
        dir=target.parent, suffix=".tmp"
    )
    try:
        with os.fdopen(fd, "w") as f:
            yield f
        os.replace(tmp_path, path)  # atomique sur POSIX
    except:
        os.unlink(tmp_path)
        raise

with atomic_write("/tmp/atomic_demo.txt") as f:
    f.write("Contenu complet et valide")

print(Path("/tmp/atomic_demo.txt").read_text())
```

</details>

---

## Ressources externes

### Documentation officielle
- [`contextlib`](https://docs.python.org/3/library/contextlib.html)
- [`with` statement](https://docs.python.org/3/reference/compound_stmts.html#the-with-statement)

### PEPs de référence
- [PEP 343 — The "with" Statement](https://peps.python.org/pep-0343/)

### Lectures complémentaires
- Fluent Python, ch. 18 « Context Managers and else Blocks »
- Effective Python, Item 66 « Consider contextlib and with Statements for Reusable try/finally Behavior »